# Dust Optical Depth (550 nm) from Sentinel-3 SYNERGY: daily series over Geneva

Alternative source to the BSC notebook. Data: **Sentinel-3 OLCI/SLSTR SYNERGY Level-2 Aerosol (SY_2_AOD)**, collection `sentinel-3-synergy-aod-l2-netcdf` on the Microsoft Planetary Computer (April 2020 to present, ~4.5 km super-pixels).

**Workflow**
1. CONFIG (period, bounding box, options)
2. STAC search on the Planetary Computer
3. Inspect one scene (check variable names)
4. Extract the bbox statistics scene by scene (resumable cache)
5. Aggregate to a daily series
6. Export CSV (`time,latitude,longitude,OD550_DUST`, same structure as the BSC output) plus a diagnostics CSV, and plot

**Things to know before comparing with BSC**
- Sentinel-3 is an *observation*, BSC is a *model*. The SYNERGY product carries a dust AOD at 550 nm, but it comes from the retrieval's aerosol-model mixture, not from an independent dust detection. Treat it as an estimate.
- Retrievals exist only for **cloud-free** super-pixels at ~10:00 local time, from one or two overpasses per day, so many days will be NaN (unlike the continuous model output of BSC).
- The Geneva bbox covers only a handful of 4.5 km super-pixels, so daily values are noisy. Check `n_pixels` before interpreting a day.
- Each NetCDF is ~150 MB. By default the notebook reads files remotely (HTTP range requests) instead of downloading them.

In [1]:
# ============================== CONFIG ==============================
DEMO_MODE = False            # True = synthetic scenes, no network / no credentials needed

START_DATE = "2025-01-01"    # inclusive (SYNERGY AOD on the Planetary Computer starts April 2020)
END_DATE   = "2025-12-31"    # inclusive

# Geneva bounding box (decimal degrees, WGS84): replace with the one used in the BSC notebook
BBOX = {"west": 5.95, "south": 46.13, "east": 6.31, "north": 46.37}

STAC_URL   = "https://planetarycomputer.microsoft.com/api/stac/v1"
COLLECTION = "sentinel-3-synergy-aod-l2-netcdf"
ASSET_KEY  = "ntc-aod"

# Variable names inside NTC_AOD.nc. None = auto-detect (see section 3 to check/override).
DUST_VAR      = None
TOTAL_AOD_VAR = None

# Pixel filter
MIN_VALUE, MAX_VALUE = 0.0, 10.0   # discard values outside this range

# Performance / robustness
STREAM_REMOTE = True         # True: read via HTTP range requests; False: download each file to /tmp
MAX_WORKERS   = 4
N_RETRIES     = 2

OUTPUT_DIR = "outputs_dod_sentinel3"

In [2]:
#%pip install pystac-client planetary-computer xarray h5netcdf netCDF4 fsspec aiohttp requests pandas matplotlib h5py
import os, re, tempfile, time, warnings
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=RuntimeWarning)

OUT = Path(OUTPUT_DIR)
OUT.mkdir(parents=True, exist_ok=True)
SCENES_CSV = OUT / f"s3_scene_stats_{START_DATE}_{END_DATE}.csv"
DAILY_CSV  = OUT / f"dod550_sentinel3_geneva_{START_DATE}_{END_DATE}.csv"        # same structure as SDE2022_mean.csv
DIAG_CSV   = OUT / f"dod550_sentinel3_geneva_{START_DATE}_{END_DATE}_diagnostics.csv"  # max, total AOD, n_scenes, n_pixels

if not DEMO_MODE:
    import pystac_client
    import planetary_computer
    import requests

assert pd.Timestamp(START_DATE) <= pd.Timestamp(END_DATE), "START_DATE after END_DATE"
if pd.Timestamp(START_DATE) < pd.Timestamp("2020-04-01"):
    print("Warning: the SYNERGY AOD collection starts in April 2020; earlier dates will return nothing.")

## 1. STAC search

In [ ]:
items = []
if DEMO_MODE:
    print("DEMO_MODE: skipping STAC search")
else:
    catalog = pystac_client.Client.open(STAC_URL)
    search = catalog.search(
        collections=[COLLECTION],
        bbox=[BBOX["west"], BBOX["south"], BBOX["east"], BBOX["north"]],
        datetime=f"{START_DATE}/{END_DATE}",
    )
    items = sorted(search.items(), key=lambda it: it.datetime or pd.Timestamp.min.tz_localize("UTC"))
    print(f"{len(items)} scenes intersect the bbox between {START_DATE} and {END_DATE}")
    if items:
        print("first:", items[0].id)
        print("last: ", items[-1].id)

## 2. Helpers: open a scene, locate variables, extract bbox statistics

In [5]:
LAT_NAMES = ["latitude", "lat", "LATITUDE", "Latitude"]
LON_NAMES = ["longitude", "lon", "LONGITUDE", "Longitude"]


def _attrs_text(var):
    return " ".join(str(var.attrs.get(k, "")) for k in ("long_name", "standard_name", "description")).lower()


def find_coord(ds, candidates):
    for n in candidates:
        if n in ds.variables:
            return n
    raise KeyError(f"None of {candidates} found; variables: {list(ds.variables)[:40]}")


def detect_dust_var(ds):
    if DUST_VAR:
        return DUST_VAR
    for n, v in ds.data_vars.items():
        if "dust" in n.lower() or "dust" in _attrs_text(v):
            if "uncert" not in n.lower() and "error" not in n.lower():
                return n
    return None


def detect_total_var(ds):
    if TOTAL_AOD_VAR:
        return TOTAL_AOD_VAR
    for n, v in ds.data_vars.items():
        low = n.lower()
        if re.fullmatch(r"(aod|aot)[_]?550", low):
            return n
    for n, v in ds.data_vars.items():
        t = (n + " " + _attrs_text(v)).lower()
        if "550" in t and ("aerosol optical" in t or "aod" in t) and not any(
            w in t for w in ("dust", "fine", "uncert", "error", "absorp", "angstrom", "ssa", "albedo", "flag")
        ):
            return n
    return None


def describe_variables(ds):
    rows = []
    for n, v in ds.variables.items():
        rows.append({"name": n, "dims": str(v.dims), "units": v.attrs.get("units", ""),
                     "long_name": v.attrs.get("long_name", v.attrs.get("description", ""))})
    return pd.DataFrame(rows)


def open_scene(item):
    """Return (dataset, cleanup_callable). Signs the asset href right before use (SAS tokens expire)."""
    href = item.assets[ASSET_KEY].href
    url = planetary_computer.sign(href)
    kw = dict(engine="h5netcdf", mask_and_scale=True, decode_times=False)
    if STREAM_REMOTE:
        import fsspec
        f = fsspec.open(url, "rb", block_size=8 * 1024 * 1024).open()
        ds = xr.open_dataset(f, **kw)
        return ds, lambda: (ds.close(), f.close())
    tmp = tempfile.NamedTemporaryFile(suffix=".nc", delete=False)
    tmp.close()
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(tmp.name, "wb") as out:
            for chunk in r.iter_content(chunk_size=1 << 20):
                out.write(chunk)
    ds = xr.open_dataset(tmp.name, **kw)
    return ds, lambda: (ds.close(), os.remove(tmp.name))


def _bbox_stats(ds, varname, mask):
    arr = ds[varname].where(mask).values.astype("float64")
    arr = arr[np.isfinite(arr)]
    arr = arr[(arr >= MIN_VALUE) & (arr <= MAX_VALUE)]
    return arr


def extract_scene(item):
    dt = item.datetime or pd.Timestamp(item.properties["start_datetime"])
    dt = pd.Timestamp(dt).tz_convert("UTC") if pd.Timestamp(dt).tzinfo else pd.Timestamp(dt).tz_localize("UTC")
    row = {"item_id": item.id, "datetime_utc": dt.isoformat(), "date": dt.strftime("%Y-%m-%d"),
           "n_pixels_bbox": 0, "n_valid_dust": 0, "dust_mean": np.nan, "dust_max": np.nan,
           "n_valid_total": 0, "total_mean": np.nan, "error": ""}
    last_err = ""
    for attempt in range(N_RETRIES + 1):
        try:
            ds, cleanup = open_scene(item)
            try:
                lat = ds[find_coord(ds, LAT_NAMES)]
                lon = ds[find_coord(ds, LON_NAMES)]
                mask = (lat >= BBOX["south"]) & (lat <= BBOX["north"]) & (lon >= BBOX["west"]) & (lon <= BBOX["east"])
                row["n_pixels_bbox"] = int(mask.sum())
                if row["n_pixels_bbox"] > 0:
                    dv = detect_dust_var(ds)
                    if dv is None:
                        raise KeyError("No dust AOD variable found; set DUST_VAR (see section 3)")
                    d = _bbox_stats(ds, dv, mask)
                    row.update(n_valid_dust=int(d.size),
                               dust_mean=float(d.mean()) if d.size else np.nan,
                               dust_max=float(d.max()) if d.size else np.nan)
                    tv = detect_total_var(ds)
                    if tv is not None:
                        t = _bbox_stats(ds, tv, mask)
                        row.update(n_valid_total=int(t.size), total_mean=float(t.mean()) if t.size else np.nan)
            finally:
                cleanup()
            return row
        except Exception as e:  # network hiccups, expired token, etc.
            last_err = f"{type(e).__name__}: {e}"
            time.sleep(2 * (attempt + 1))
    row["error"] = last_err
    return row

## 3. Inspect one scene (check variable names)

Run this once. If the auto-detection in section 2 picks the wrong variables, set `DUST_VAR` / `TOTAL_AOD_VAR` in CONFIG.

In [ ]:
if DEMO_MODE or not items:
    print("Nothing to inspect (demo mode or no scenes).")
else:
    ds0, cleanup0 = open_scene(items[0])
    try:
        variables = describe_variables(ds0)
        display(variables)
        print("Auto-detected dust variable :", detect_dust_var(ds0))
        print("Auto-detected total AOD var :", detect_total_var(ds0))
    finally:
        cleanup0()

## 4. Extract bbox statistics scene by scene

Results are appended to a scene-level CSV, so an interrupted run resumes where it stopped.

In [ ]:
def make_demo_scenes(start, end, seed=42):
    """Synthetic scenes mimicking sparse, cloud-limited retrievals with occasional dust events."""
    rng = np.random.default_rng(seed)
    rows = []
    for day in pd.date_range(start, end, freq="D"):
        if rng.random() < 0.55:          # cloud-free enough for a retrieval
            n_scenes = rng.integers(1, 3)
            event = 1.0 + (rng.random() < 0.04) * rng.uniform(3, 10)
            for k in range(n_scenes):
                n = int(rng.integers(0, 14))
                dt = day + pd.Timedelta(hours=9 + k, minutes=int(rng.integers(0, 59)))
                base = rng.lognormal(mean=np.log(0.02), sigma=0.6) * event
                rows.append({"item_id": f"DEMO_{dt:%Y%m%dT%H%M}", "datetime_utc": dt.isoformat() + "+00:00",
                             "date": dt.strftime("%Y-%m-%d"), "n_pixels_bbox": 14, "n_valid_dust": n,
                             "dust_mean": base if n else np.nan, "dust_max": base * 1.6 if n else np.nan,
                             "n_valid_total": n, "total_mean": base * 6 + 0.05 if n else np.nan, "error": ""})
    return pd.DataFrame(rows)


if DEMO_MODE:
    scenes = make_demo_scenes(START_DATE, END_DATE)
    print(f"DEMO_MODE: {len(scenes)} synthetic scenes")
else:
    done = pd.read_csv(SCENES_CSV) if SCENES_CSV.exists() else pd.DataFrame()
    done_ok = set(done.loc[done["error"].fillna("") == "", "item_id"]) if len(done) else set()
    todo = [it for it in items if it.id not in done_ok]
    print(f"{len(done_ok)} scenes already cached, {len(todo)} to process")

    new_rows = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = {pool.submit(extract_scene, it): it.id for it in todo}
        for i, fut in enumerate(as_completed(futures), 1):
            new_rows.append(fut.result())
            if i % 25 == 0 or i == len(futures):
                print(f"  {i}/{len(futures)} scenes", end="\r")
                pd.concat([done, pd.DataFrame(new_rows)], ignore_index=True).drop_duplicates(
                    "item_id", keep="last").to_csv(SCENES_CSV, index=False)
    scenes = pd.read_csv(SCENES_CSV) if SCENES_CSV.exists() else pd.DataFrame(columns=["item_id"])
    n_err = int((scenes.get("error", pd.Series(dtype=str)).fillna("") != "").sum())
    print(f"\nDone. {len(scenes)} scenes, {n_err} with errors (re-run this cell to retry them).")

scenes.head()

## 5. Daily aggregation

Daily value = pixel-weighted mean of all valid super-pixels inside the bbox, over all scenes of the UTC day. Days without any valid pixel are NaN.

In [ ]:
sc = scenes.copy()
sc["date"] = pd.to_datetime(sc["date"])
sc["n_valid_dust"] = sc["n_valid_dust"].fillna(0)
sc["n_valid_total"] = sc["n_valid_total"].fillna(0)
sc["w_dust"] = sc["dust_mean"].fillna(0) * sc["n_valid_dust"]
sc["w_total"] = sc["total_mean"].fillna(0) * sc["n_valid_total"]
sc["has_pixels"] = (sc["n_valid_dust"] > 0).astype(int)

g = sc.groupby("date").agg(
    n_scenes=("has_pixels", "sum"),
    n_pixels=("n_valid_dust", "sum"),
    dust_sum=("w_dust", "sum"),
    dust_max=("dust_max", "max"),
    n_total=("n_valid_total", "sum"),
    total_sum=("w_total", "sum"),
)
daily = pd.DataFrame(index=g.index)
daily["dod_550"] = np.where(g["n_pixels"] > 0, g["dust_sum"] / g["n_pixels"].replace(0, np.nan), np.nan)
daily["dod_550_max"] = g["dust_max"].where(g["n_pixels"] > 0)
daily["aod_550_total"] = np.where(g["n_total"] > 0, g["total_sum"] / g["n_total"].replace(0, np.nan), np.nan)
daily["n_scenes"] = g["n_scenes"].astype(int)
daily["n_pixels"] = g["n_pixels"].astype(int)

full_index = pd.date_range(START_DATE, END_DATE, freq="D", name="date")
daily = daily.reindex(full_index)
daily[["n_scenes", "n_pixels"]] = daily[["n_scenes", "n_pixels"]].fillna(0).astype(int)

print(f"{daily['dod_550'].notna().sum()} of {len(daily)} days have a valid DOD value "
      f"({100 * daily['dod_550'].notna().mean():.0f}%)")
daily.describe()

## 6. Export and plot

Main CSV has the same columns as the BSC output (`time, latitude, longitude, OD550_DUST`); latitude/longitude are the bbox centre. Days without data are written as `date,,,`. Extra columns go to a separate `_diagnostics.csv`.

In [ ]:
def write_reference_csv(series, path):
    """Write time,latitude,longitude,OD550_DUST (one row per day; missing days as 'date,,,')."""
    lat_c = 0.5 * (BBOX["south"] + BBOX["north"])
    lon_c = 0.5 * (BBOX["west"] + BBOX["east"])
    out = pd.DataFrame({"time": series.index.strftime("%Y-%m-%d"),
                        "latitude": lat_c, "longitude": lon_c,
                        "OD550_DUST": series.values})
    out.loc[out["OD550_DUST"].isna(), ["latitude", "longitude"]] = np.nan
    out.to_csv(path, index=False, float_format="%.8g")


write_reference_csv(daily["dod_550"], DAILY_CSV)
daily.to_csv(DIAG_CSV, float_format="%.5f")
print("Saved", DAILY_CSV, "and", DIAG_CSV)

fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
ax[0].plot(daily.index, daily["dod_550"], "o", ms=3, color="tab:orange", label="DOD 550 nm (Sentinel-3 SYNERGY)")
if daily["aod_550_total"].notna().any():
    ax[0].plot(daily.index, daily["aod_550_total"], ".", ms=2, color="0.6", label="Total AOD 550 nm")
ax[0].set_ylabel("Optical depth (-)")
ax[0].set_title("Sentinel-3 SYNERGY DOD at 550 nm, Geneva bbox" + (" [DEMO DATA]" if DEMO_MODE else ""))
ax[0].legend(loc="upper right")
ax[0].grid(alpha=0.3)
ax[1].bar(daily.index, daily["n_pixels"], width=1.0, color="0.4")
ax[1].set_ylabel("valid\npixels")
ax[1].grid(alpha=0.3)
plt.tight_layout()
fig.savefig(OUT / f"dod550_sentinel3_geneva_{START_DATE}_{END_DATE}.png", dpi=150)
plt.show()